# Data Aggregation and Analysis
Hello,
I have provided you with a data set named “supermarket_transactions.csv.” This data set contains three years of transactional data we collected from supermarkets across Australia. 
InsightSpark wants to access our data to perform some analysis across Australian supermarkets. Before this can happen, I need you to perform some analysis on the provided CSV data set (a sample of the database) to answer the following questions:
* Across locations, how many apples were purchased in cash?
* How much total cash was spent on these apples?
* Across all payment methods, how much money was spent at the Bakershire store location by non-member customers?


Import relevant libs

In [ ]:
import numpy as np
import pandas as pd
import sqlite3

Read as dataframe and establish `sql` connection

In [5]:
# read the xlsx file and set the first column as the index
df = pd.read_excel('../1.3.Task/supermarket_transactions.xlsx', index_col=0)

## Sql connection

In [38]:
# A temp dir to store the db file
import tempfile, os.path
dbfile = os.path.join(tempfile.mkdtemp(), "supermarket_transactions.db")
print("Temp db file saved into ",dbfile)

# sql connection
conn = sqlite3.connect(dbfile)
print("------\nDatabase connection established")

Temp db file saved into  C:\Users\HP\AppData\Local\Temp\tmp0qymr7hd\supermarket_transactions.db
------
Database connection established


In [ ]:
df.to_sql('transactions',index=False, con=conn)
pd.read_sql('SELECT * FROM transactions LIMIT 3', con=conn)

,id,timestamp,quantity,product_id,product_name,unit_price,total_amount,store,payment_method,customer_id,customer_type
0,47d54138-a950-4ec0-9d4a-e637e8dfb290,2022-04-28 09:11:00,10,56b274d1-b2a4-47be-abaf-6c1c7742beb9,wheat,0.39,3.90,South Billyview,cash,c2b9eddd-df8c-446a-ae34-50ad88f28e0a,non-member
1,dea298b1-26ca-4a43-bcef-4050fb74ce1d,2019-12-06 16:21:00,1,56b274d1-b2a4-47be-abaf-6c1c7742beb9,wheat,0.39,0.39,South Edward,contactless,134a3c2c-8ccc-494c-9916-a15ff10a9125,member
2,8dcd78f6-2e68-4fe3-9bb0-afe3bb2c3944,2019-06-22 15:27:00,10,56b274d1-b2a4-47be-abaf-6c1c7742beb9,wheat,0.39,3.90,South Billyview,contactless,a310b698-4e34-4a4a-8c60-e72fad63f9ba,gold


## Prior Analysis
Before answering to the 3 questions, let's preprocess the dataset:

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50783 entries, 0 to 50782
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              50783 non-null  object        
 1   timestamp       50783 non-null  datetime64[ns]
 2   quantity        50783 non-null  int64         
 3   product_id      50783 non-null  object        
 4   product_name    50783 non-null  object        
 5   unit_price      50783 non-null  float64       
 6   total_amount    50783 non-null  float64       
 7   store           50783 non-null  object        
 8   payment_method  50783 non-null  object        
 9   customer_id     50783 non-null  object        
 10  customer_type   50783 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(7)
memory usage: 4.6+ MB


In [21]:
df.head(3)
df.nunique()
df.describe()
df['product_name'].value_counts().tail(10)

product_name
Parmesan        76
flatfish        76
garlic sauce    75
Swiss cheese    75
muesli          75
green tea       75
pecans          75
squid           75
cookies         75
sole            75
Name: count, dtype: int64

## Q1. Number of Apples Purchase by Cash
Across locations, how many apples were purchased in cash?

In [35]:
#print the unique number of values of 'store' and 'payment_method' columns
print("Total stores:", df['store'].nunique())
print("Number of different payment methods:", df['payment_method'].nunique())
df['payment_method'].value_counts()

Total stores: 48
Number of different payment methods: 4


payment_method
contactless    12781
cash           12730
debit card     12658
credit card    12614
Name: count, dtype: int64

In [100]:
# apples sold per store with cash payment
temp = '''
SELECT store, SUM(quantity) as cash_sales_apple_count \
    FROM transactions \
        WHERE payment_method="cash" AND product_name="apple" \
            GROUP BY store ORDER BY cash_sales_apple_count DESC
'''

# total apples sold with cash payment
q1 = '''
SELECT SUM(quantity) as total_apples \
    FROM transactions \
        WHERE product_name="apple" AND payment_method="cash"\
'''

# run the queries and store the results in dataframes
total_apples = pd.read_sql(sql=q1, con=conn)
apples_sold_per_store = pd.read_sql(sql=temp, con=conn)

print("Total number of apples sold with cash:", total_apples['total_apples'][0])
apples_sold_per_store

Total number of apples sold with cash: 117


,store,cash_sales_apple_count
0,South Cynthia,18
1,Swansonfurt,17
2,South Billyview,12
3,South Michaelfurt,10
4,East Suzanneside,10
5,South Edward,8
6,Lake Bryan,8
7,Tracyton,7
8,Erichaven,7
9,Alexmouth,6


## Q2. Apples sold with cash
How much total cash was spent on these apples?

In [93]:
temp1 = '''
SELECT timestamp,store, product_name, quantity, total_amount, unit_price, payment_method \
    FROM transactions \
        WHERE product_name="apple" AND store="South Billyview" \
            ORDER BY quantity DESC
    '''

temp = '''
SELECT store, product_name, quantity, total_amount, unit_price, payment_method \
    FROM transactions \
        WHERE product_name="apple" AND payment_method="cash" \
            ORDER BY quantity DESC
    '''
# pd.read_sql(sql=temp, con=conn)

In [98]:
# amount of cash sales per store for apples
temp2 = '''
SELECT store, SUM(total_amount) as total_cash_sales \
    FROM transactions \
        WHERE product_name="apple" AND payment_method="cash" \
            GROUP BY store
            ORDER BY total_cash_sales DESC
    '''
apples_cash_sales_by_store = pd.read_sql(sql=temp2, con=conn)


# total amount of cash sales for apples
q2 = '''
SELECT SUM(total_amount) as total_cash_sales \
    FROM transactions \
        WHERE product_name="apple" AND payment_method="cash"
    '''
apples_cash_sales = pd.read_sql(sql=q2, con=conn)
print(f"Total amount of apples sold with cash: ${apples_cash_sales['total_cash_sales'][0]:.2f}")

apples_cash_sales_by_store

Total amount of apples sold with cash: $537.03


,store,total_cash_sales
0,South Cynthia,82.62
1,Swansonfurt,78.03
2,South Billyview,55.08
3,South Michaelfurt,45.90
4,East Suzanneside,45.90
5,South Edward,36.72
6,Lake Bryan,36.72
7,Tracyton,32.13
8,Erichaven,32.13
9,Alexmouth,27.54


## Q3. Sales at Bakershire
Across all payment methods, how much money was spent at the *Bakershire* store location by *non-member* customers?

In [114]:
print("Types of customers: ",df['customer_type'].nunique())

df['customer_type'].value_counts(normalize=True)
df['customer_type'].value_counts()

Types of customers:  6


customer_type
non-member    8650
corporate     8536
premium       8427
staff         8397
member        8395
gold          8378
Name: count, dtype: int64

In [134]:
# Total sales at Bakershire by payment method
temp = '''
SELECT payment_method, SUM(total_amount) as total_sales \
    FROM transactions \
        WHERE store="Bakershire" \
            GROUP BY payment_method
            ORDER BY total_sales DESC
    '''
print("Total sales at Bakershire:", pd.read_sql(sql=temp, con=conn).sum()['total_sales'])
pd.read_sql(sql=temp, con=conn)


Total sales at Bakershire: 15838.019999999997


,payment_method,total_sales
0,cash,4086.28
1,credit card,4057.74
2,debit card,3848.85
3,contactless,3845.15


### non-members
The goal is **not** to find the customers that are **non-member**, but the total amount of sales through those customers:
> NOTE: apple is not mentioned/ therefore, all the products.

In [156]:
# Total sales at Bakershire by customer type
temp = '''
SELECT customer_type, SUM(total_amount) as total_sales \
    FROM transactions \
        WHERE store="Bakershire" \
            GROUP BY customer_type
            ORDER BY total_sales
    '''
all_customers = pd.read_sql(sql=temp, con=conn)
print("Total sales at Bakershire by customer type:\n",all_customers) 
print("_"*40)
print(f"Total sales at Bakershire : ${all_customers.sum()['total_sales']:,.2f}")

Total sales at Bakershire by customer type:
   customer_type  total_sales
0          gold      2388.92
1         staff      2517.04
2       premium      2604.98
3        member      2696.72
4     corporate      2772.85
5    non-member      2857.51
________________________________________
Total sales at Bakershire : $15,838.02


In [ ]:
# total sales at Bakershire through non-member customers
q3 = '''
SELECT SUM(total_amount) \
    FROM transactions \
        WHERE store="Bakershire" AND \
            customer_type="non-member"
    '''
non_member_sales = pd.read_sql(sql=q3, con=conn)
print(f"\nSales at Bakershire through `non-member` customers: ${non_member_sales['SUM(total_amount)'][0]:,.2f}")